<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/Daychallenge_W8_J3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Installation des dépendances
Nous installons la bibliothèque `mcp` avec les outils CLI nécessaires.

In [ ]:
!pip install -U "mcp[cli]"

### 2. Création de `server.py`
Ce script définit le serveur `FastMCP` avec un outil (`tool`) et une ressource (`resource`).

In [ ]:
%%writefile server.py
import logging
from mcp.server.fastmcp import FastMCP

# Configuration de la journalisation sur stderr pour ne pas polluer stdout (utilisé par le transport)
logging.basicConfig(level=logging.INFO)

# Initialisation du serveur FastMCP
mcp = FastMCP("WeatherDemo")

# Base de données en mémoire
WEATHER_DATA = {
    "Paris": {"temp_c": 21, "condition": "Sunny"},
    "London": {"temp_c": 18, "condition": "Cloudy"},
    "New York": {"temp_c": 27, "condition": "Rainy"}
}

@mcp.tool()
def get_weather(city: str) -> dict:
    """Obtient les conditions météo pour une ville donnée."""
    logging.info(f"Appel de l'outil get_weather pour : {city}")
    data = WEATHER_DATA.get(city)
    if data:
        return {"city": city, **data}
    return {"error": "City not found"}

@mcp.resource("cities://list")
def list_cities() -> str:
    """Retourne la liste des villes supportées."""
    logging.info("Accès à la ressource cities://list")
    return "\n".join(WEATHER_DATA.keys())

if __name__ == "__main__":
    # Démarrage du serveur sur STDIO
    mcp.run()

### 3. Création de `client.py`
Ce script agit comme l'hôte qui lance le serveur, inspecte ses capacités et invoque ses fonctions.

In [ ]:
%%writefile client.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def run_client():
    # Configuration du transport vers le serveur
    server_params = StdioServerParameters(
        command="mcp",
        args=["run", "server.py"],
        env=None
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # 1. Initialisation de la session
            await session.initialize()

            # 2. Découverte des ressources
            print("\nResources:")
            resources = await session.list_resources()
            for resource in resources.resources:
                print(f"- {resource.name}")

            # 3. Découverte des outils
            print("\nTools:")
            tools = await session.list_tools()
            for tool in tools.tools:
                print(f"- {tool.name}")

            # 4. Lecture d'une ressource
            print("\nReading cities://list")
            content = await session.read_resource("cities://list")
            print(content.contents[0].text)

            # 5. Appel d'un outil
            print("\nCalling get_weather(\"Paris\")")
            result = await session.call_tool("get_weather", arguments={"city": "Paris"})
            # FastMCP retourne souvent le résultat dans le premier item de 'content'
            print(result.content[0].text)

if __name__ == "__main__":
    asyncio.run(run_client())

### 4. Exécution du projet
Nous lançons le client, qui va automatiquement démarrer `server.py` en arrière-plan.

In [ ]:
!python client.py

### 5. Explications Théoriques

- **Qu'est-ce que MCP ?** Le *Model Context Protocol* est un standard ouvert permettant d'unifier la manière dont les applications (souvent des IA) accèdent aux données et aux outils externes.
- **Rôle du Serveur :** Le serveur expose des capacités (outils, ressources) de manière structurée sans savoir qui va les consommer. Il attend des commandes sur son canal de communication.
- **Rôle du Client :** Le client est l'orchestrateur. Il se connecte au serveur, découvre ce qu'il sait faire (introspection) et lui envoie des requêtes d'exécution.
- **Pourquoi STDIO ?** C'est le transport le plus simple : le client lance le processus du serveur et communique via les flux standard d'entrée/sortie (`stdin`/`stdout`). C'est idéal pour des agents locaux.
- **Tool vs Resource :**
    - Une **Resource** est une donnée passive (comme un fichier ou une liste) que le client peut lire.
    - Un **Tool** est une fonction active qui peut prendre des paramètres et effectuer une action ou un calcul.
- **Flux de données :** Le client envoie un message JSON-RPC sur le `stdin` du serveur. Le serveur traite la demande et renvoie la réponse en JSON-RPC sur son `stdout`, que le client intercepte.